# 03 — Full benchmark

Run every case in `fda-strategy-triples` v0.1.0 through the stack, regenerate the scorecard, and render the results inline. This notebook is the source of truth for the table at the top of the README — it is what `scripts/run_e2e.sh` does, but with all the intermediate state visible.

In [ ]:
from pathlib import Path
from tqdm import tqdm

from fda_strategy_triples import load_cases
from g2p_rag import Retriever
from therapy_agent import TherapyAgent
from bio_rag_eval import score_case, render_scorecard

retriever = Retriever.from_pretrained('v0.1.0')
agent = TherapyAgent(model='claude-opus-4-7')

cases = list(load_cases())
print(f'Loaded {len(cases)} cases')

In [ ]:
scores = []
for case in tqdm(cases, desc='Running stack'):
    context = retriever.retrieve(case.disease_gene, k=10)
    hypotheses = agent.propose(case.disease_gene, context)
    scores.append(score_case(case, hypotheses))

In [ ]:
import statistics
recovered = sum(s.recovered for s in scores)
mean_judge = statistics.mean(s.judge_score for s in scores)
mean_rank = statistics.mean(s.rank for s in scores if s.rank is not None)
print(f'Recovered: {recovered}/{len(scores)}')
print(f'Mean judge score: {mean_judge:.3f}')
print(f'Mean rank of correct target: {mean_rank:.2f}')

In [ ]:
out = Path('../assets/scorecard_v0.1.0.html')
render_scorecard(scores, out)
print(f'Wrote {out.resolve()}')

In [ ]:
from IPython.display import IFrame
IFrame(src='../assets/scorecard_v0.1.0.html', width=900, height=600)

## Inspecting failures

If any case fails to recover, surface it here for debugging. A failure where the agent confidently proposes the wrong target is more interesting than one where it lists the right target at rank 3.

In [ ]:
for s in scores:
    if not s.recovered:
        print(f'FAIL  {s.case_id}: predicted top={s.top_prediction.target}, gold={s.gold_target}')
        print(f'      rationale: {s.top_prediction.rationale[:200]}...')
        print()
    elif s.rank > 1:
        print(f'SOFT  {s.case_id}: gold={s.gold_target} at rank {s.rank}')